### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import combinations
from astropy.constants import si as constants
from astropy import units as u
from astropy.table import Table
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm 
from tabulate import tabulate
from survey_tools import sky
from ao_tools import etc

### Options

In [ ]:
lines = [
    {'name': 'Hb'  , 'wavelength_vac': np.array([0.4862721           ]), 'dispersion': 75.0, 'flux_ratio': np.array([0.350       ]), 'snr_threshold': np.array([1.0     ])},  # 4862.721 Å
    {'name': 'OIII', 'wavelength_vac': np.array([0.5008240, 0.4960295]), 'dispersion': 75.0, 'flux_ratio': np.array([1.120, 0.380]), 'snr_threshold': np.array([3.0, 1.0])},  # 5008.240, 4960.295 Å (stronger-first)
   #{'name': 'OI'  , 'wavelength_vac': np.array([0.6300304, 0.6363776]), 'dispersion': 75.0, 'flux_ratio': np.array([0.015, 0.005]), 'snr_threshold': np.array([1.0, 1.0])},  # 6300.304, 6363.776 Å (stronger-first)
    {'name': 'NII' , 'wavelength_vac': np.array([0.6585270, 0.6549860]), 'dispersion': 75.0, 'flux_ratio': np.array([0.187, 0.063]), 'snr_threshold': np.array([1.0, 1.0])},  # 6585.270, 6549.860 Å (stronger-first)
    {'name': 'Ha'  , 'wavelength_vac': np.array([0.6564614           ]), 'dispersion': 75.0, 'flux_ratio': np.array([1.000       ]), 'snr_threshold': np.array([3.0     ])},  # 6564.614 Å
   #{'name': 'SII' , 'wavelength_vac': np.array([0.6732670, 0.6718290]), 'dispersion': 75.0, 'flux_ratio': np.array([0.105, 0.095]), 'snr_threshold': np.array([1.0, 1.0])},  # 6732.68, 6718.29 Å (trength varies with n_e)
]

include_weaker_lines = True

R            = 3000  # 3000 or 8000
fov          = 2.0 * u.arcsec
N_exp        = 24             # number of exposures
T_exp        = 600.0 * u.s    # exposure time
nPix         = 4              # number of pixels in aperture
eta_spat     = 0.01           # fraction of light in aperture
flux         = 2e-16 * u.erg/u.s/u.cm**2  # line flux to use
zenith_angle = 20 * u.deg    # zenith angle

### Prepare

In [ ]:
wvl_bands = etc.get_wvl_bands('GIRMOS', R)
wvl_min = wvl_bands[0][1]
wvl_max = wvl_bands[-1][2]
print(f"Bands: {wvl_min.to_value(u.micron)}-{wvl_max.to_value(u.micron)} μm (R={R})")

### Calculations

In [ ]:
line_names = [l['name'] for l in lines]

if not include_weaker_lines:
    for name in ['OIII', 'OI', 'NII']:
        if name in line_names:
            idx = line_names.index(name)
            lines[idx]['wavelength_vac'] = lines[idx]['wavelength_vac'][0:1]

for line in lines:
    if 'wavelength_vac' in line:
        line['wavelength'] = sky.get_vacuum_to_air_wavelength(line['wavelength_vac']*u.micron).value
    if 'wavelength' not in line:
        raise ValueError(f"Line {line['name']} does not have 'wavelength' defined.")

In [ ]:
min_wvl_rest = np.min([np.min(line['wavelength']) for line in lines]) * u.micron
max_wvl_rest = np.max([np.max(line['wavelength']) for line in lines]) * u.micron

dlambda = np.round(wvl_min / R, 5)
N = int((wvl_max - wvl_min) / dlambda)

min_redshift = np.round(wvl_min.to_value(u.micron) / min_wvl_rest.to_value(u.micron) - 1, 5)
max_redshift = np.round(wvl_max.to_value(u.micron) / max_wvl_rest.to_value(u.micron) - 1, 5)
dz = np.round((max_redshift - min_redshift) / N, 5)
min_redshift = np.ceil(min_redshift / dz) * dz  # Adjust min_redshift so N dz's fit in range
max_redshift = min_redshift + dz * (N-1) # Adjust max_redshift so N dz's fit in range
redshifts = np.linspace(min_redshift, max_redshift, N)
print(f"Redshift Range to Search: {min_redshift:.3f}-{max_redshift:.3f}, N={N}, dz = {dz:.5f}")

In [ ]:
usable_redshifts = np.ones_like(redshifts, dtype=bool)
redshift_num_bands = np.zeros_like(redshifts, dtype=int)

plotted = False
num_plotable = 0

etc_params = {
    'fov': fov,
    'R': R,
    'bands': wvl_bands,
    'T_exp': T_exp,
    'N_exp': N_exp,
    'nPix': nPix
}

for i, z in enumerate(redshifts):
    line_bands = np.zeros((len(lines), len(wvl_bands)), dtype=np.bool)

    for j, line in enumerate(lines):
        wvl = line['wavelength'] * (1 + z) * u.micron
        sigma_v = line['dispersion'] * u.km / u.s
        sigma = (sigma_v / constants.c * wvl).to(u.micron)
        flux_ratio = line['flux_ratio']
        snr_threshold = line['snr_threshold']

        rejects = sky.reject_emission_line(
            etc_params,
            flux * flux_ratio, wvl, sigma_v, eta_spat, 
            snr_threshold=snr_threshold
        )

        if np.all(rejects):
            usable_redshifts[i] = False
        else:
            for k in range(len(wvl)):
                if not rejects[k]:
                    for l, band in enumerate(wvl_bands):
                        if band[1] <= wvl[k] and wvl[k] <= band[2]:
                            line_bands[j, l] = True

    if usable_redshifts[i]:
        min_bands = None
        for num_cols in range(1, line_bands.shape[1] + 1):
            for band_indices in combinations(range(line_bands.shape[1]), num_cols):
                if np.all(np.any(line_bands[:, band_indices], axis=1)):
                    min_bands = num_cols
                    break
            if min_bands is not None:
                break
        redshift_num_bands[i] = min_bands if min_bands is not None else 0

        if not plotted:
            num_plotable += 1
            if num_plotable < 100:
                continue

            plot_wavelength_range = [wvl[0]-20*np.max(sigma), wvl[0]+20*np.max(sigma)]
            plot_wvl = np.linspace(plot_wavelength_range[0], plot_wavelength_range[1], 1000)

            rejects, ENBW, N_signal, N_noise = sky.reject_emission_line(
                etc_params,
                flux * flux_ratio, wvl, sigma_v, eta_spat, 
                snr_threshold=snr_threshold,
                return_ENBW=True,
                return_signals=True,
                spec_wvl=plot_wvl,
            )

            etc.plot_estimated_signals(plot_wvl, wvl, N_signal, N_noise, ENBW, rejects)
            plotted = True

In [ ]:
print(f"Usable Redshifts: N={np.sum(usable_redshifts)}/{len(redshifts)} ({np.sum(usable_redshifts)/N:.1%})")
print("Usable Redshifts by Num Bands Required:")
for i in range(len(wvl_bands)):
    print(f"  {i+1}: {np.sum(redshift_num_bands == i+1)}")

plt.figure(figsize=(8, 4))

for i in range(len(wvl_bands)):
    if np.sum(redshift_num_bands == i+1) > 0:
        match i+1:
            case 1:
                color = 'b'
            case 2:
                color = 'g'
            case 3:
                color = 'r'
        Z = redshifts[usable_redshifts & (redshift_num_bands == i+1)]
        plt.plot(Z, np.ones_like(Z), '|', markersize=10, color=color, label=f'{i+1} band{"s" if i+1>1 else ""} (N={len(Z)})')

plt.legend()
plt.xlabel('Redshift')
plt.yticks([])
plt.title(f"Usable Redshifts for BPT with GIRMOS (R={R})")
plt.show()

In [ ]:
redshift_summary = Table({
    'Redshift': np.round(redshifts[usable_redshifts],6),
    'NumBands': redshift_num_bands[usable_redshifts]
})

suffix = "_".join(line_names)
redshift_summary.write(f"../output/bpt_redshifts_{suffix}.txt", format='ascii', overwrite=True)

display(tabulate(redshift_summary, headers=redshift_summary.colnames, tablefmt='html'))